In [ ]:
#Tener en un fichero csv, la tabla de frecuencias por Country y poder ver tanto los valores absolutos como los relativos. Necesitare usar la función merge que combina 2 dataframes distintos (pero necesitamos un campo comun para hacer match)
#0.- Llamar a las librerias
import pandas as pd
#1.- Cargar los 3 dataframes
january_2019 = pd.read_csv(r"C:\Users\formacio\Downloads\January 2019 (1).csv")
february_2019 = pd.read_csv(r"C:\Users\formacio\Downloads\February 2019 (1).csv")
march_2019 = pd.read_csv(r"C:\Users\formacio\Downloads\March 2019 (1).csv")
#2.- Concatenar los 3 dataframes
trimestre_2019 = pd.concat([january_2019,february_2019,march_2019],ignore_index=True)
#3.- Tabla de frecuencias
#Guardar en un csv las frecuencias por Country del dataframe del trimestre, tanto las absolutas como las relativas
#4.- Guardar en un dataframe las absolutas
absolutas = trimestre_2019['Country'].value_counts().reset_index()
#5.- Guardo en un dataframe las relativas
relativas = round(trimestre_2019['Country'].value_counts(normalize=True) * 100, 2).reset_index()
#6.- Combinar los 2 dataframes para tener uno solo, eso lo haremos con merge
total_frecuencias = pd.merge(absolutas,relativas,on="Country",how='inner')
total_frecuencias
#7.- Lo voy a guardar en un excel, para trabajar con excel necesito instalar aparte una libreria, que no hace falta llamarla, se llama openpyxl
total_frecuencias.to_excel(r"C:\temp\frecuencias_abs_rel.xlsx",sheet_name="Datos",index=False,header=True)
#8.- Mensaje final
print("la tabla de frecuencias ha sido guardado a excel con éxito")

In [ ]:
!pip install openpyxl

Practica 31: Mediante el fichero de excel "Datos Pedidos.xlsx", transformar los datos para poder resolver los siguientes enunciados
- Aplicar un 3.65% de descuento para todos los pedidos que hayan sido realizados una semana antes del final del mes (Campo Importe Neto). Si no es una semana antes Importe Neto = Importe
- Ver en un texto la suma de importe de los Pedidos de Alemania + USA + Brazil
- Crear un copia del dataframe (Datos Pedidos II), que contenga los pedidos cuya categoría sea Bebidas, Lacteos o Reposteria y que ademas el importe neto del pedido sea mayor de 100
- Sobre este ultimo DataFrame (Datos Pedidos II), crear una tabla de frecuencias para ver el % de frecuencia de cada Comercial (Queremos ver 20.85%)
- Guardar esta tabla de frecuencias relativas por comercial en un fichero de excel.

In [ ]:
#Por defecto pandas siempre coge la primera hoja de excel, sin importar el nombre, si tiene que ser de una especifica, ajustar el nombre con sheet_name
datos_pedidos = pd.read_excel(r"C:\Users\formacio\Downloads\Datos Pedidos.xlsx")
datos_pedidos

In [ ]:
#Solución practica 31
#0.- importar librerias
import pandas as pd
from datetime import date, datetime, timedelta
#1.- cargar un dataframe, si hay +1 hoja el parametro sheet_name="Hoja3"
datos_pedidos = pd.read_excel(r"C:\Users\formacio\Downloads\Datos Pedidos.xlsx", sheet_name="Datos")
#2.- Asegurarme que la columna/campo "Fecha de Pedido" es datetime
datos_pedidos['Fecha de Pedido'] = pd.to_datetime(datos_pedidos['Fecha de Pedido'])
#3.- Crear una función para aplicar en ambito generico y que me devuelva el Importe Neto
def importe_neto(fila):
    #Fila es el dataframe de la fila que llama a la función, la función sera
    #llamada tantas veces como filas hayan
    fecha = fila['Fecha de Pedido']
    if fila['Fecha de Pedido'].month == 12:
        #Me montara siempre el 01/01/del año siguiente
        fin_mes = date(fila['Fecha de Pedido'].year + 1, 1, 1) + timedelta(days=-1)
    else:
        fin_mes = date(fila['Fecha de Pedido'].year,fila['Fecha de Pedido'].month + 1,1) + timedelta(days=-1)
    #Averiguar el numero de dias de diferencia entre fecha de Pedido y Fin Mes
    dias_diferencia = (pd.to_datetime(fin_mes) - fecha).days
    #Con days me los da numerico
    if dias_diferencia<=7:
        return fila['Importe'] * (1 - (3.65/100))
    else:
        return fila['Importe']
#Crear la columna Importe neto
datos_pedidos['Importe Neto'] = round(datos_pedidos.apply(importe_neto,axis=1),2)
#Ver en un texto la suma de importe de los Pedidos de Alemania + USA + Brazil
filtro_paises = ['Alemania','Estados Unidos','Brasil']
filtro = datos_pedidos['Pais'].isin(filtro_paises)
print("la suma de importe de Alemania+USA+Brasil es:",datos_pedidos[filtro].Importe.sum())
#Crear un copia del dataframe (Datos Pedidos II), que contenga los pedidos cuya categoría sea Bebidas, Lacteos o Reposteria y que ademas el importe neto del pedido sea mayor de 100
filtro_categorias = ['Bebidas','Lácteos','Repostería']
filtro = (datos_pedidos['Categoria'].isin(filtro_categorias) & (datos_pedidos['Importe Neto']>100))
datos_pedidos_2 = datos_pedidos[filtro]
datos_pedidos_2['Importe Neto'].sum()
#Sobre este ultimo DataFrame (Datos Pedidos II), crear una tabla de frecuencias para ver el % de frecuencia de cada Comercial (Queremos ver 20.85%)
frecuencias = round((datos_pedidos_2['Comercial'].value_counts(normalize=True) * 100),2).reset_index()
#Función para el %
def porcentaje(valor):
    return str(valor)+'%'
frecuencias['proportion'] = frecuencias['proportion'].apply(porcentaje)
#Guardar en un excel
frecuencias.to_excel(r"C:\temp\frecuencias_comerciales.xlsx",sheet_name="Datos")